In [0]:
%run ../commonsetup/Setup-Common

##### Downloads the TfNSW GTFS static feed into the landing volume as a scheduled Workflow task -- same orchestrator as Bronze/Silver, so no second tool (ADF) just for an HTTP GET.
##### Lands files as `entity_YYYYMMDD.ext`, the convention `02a_Bronze_batch_ingestion.py`'s `discover_entity()` expects.

In [0]:
import io
import time
import zipfile
from datetime import date
 
import requests

### 1: Config
##### CONFIRM `GTFS_STATIC_URL_TEMPLATE` against your registered TfNSW Open Data Hub modes. `GTFS_MODES` drives the loop -- each mode lands independently, one mode's feed shape has no bearing on another's.

In [0]:
GTFS_STATIC_URL_TEMPLATE = "https://api.transport.nsw.gov.au/v1/gtfs/schedule/{mode}"  # CONFIRM per-mode path
 
REQUEST_TIMEOUT_SECONDS = 60
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 10
 
# Secret fetched here, at the point of use -- not in Setup-Common, which every
# notebook %runs and shouldn't unconditionally pull credentials it may not need.
api_key = dbutils.secrets.get(scope="tfnsw_secrets", key="tfnsw-api-key")
 
today_stamp = date.today().strftime("%Y%m%d")

### 2: Download with retry
##### Bounded backoff -- a scheduled job shouldn't hard-fail on one transient network blip.

In [0]:
def download_gtfs_zip(url: str, api_key: str) -> bytes:
    headers = {"Authorization": f"apikey {api_key}"}
 
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
            response.raise_for_status()
            return response.content
        except requests.exceptions.RequestException as e:
            last_error = e
            print(f"Attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)
 
    raise RuntimeError(f"GTFS download failed after {MAX_RETRIES} attempts: {last_error}")

### 3: Download + unpack per mode
##### Each mode isolated in its own try/except, same as Bronze's per-entity isolation. Skips an entity already landed for today (idempotent re-run) and doesn't error on an optional file the feed omits (e.g. shapes.txt) -- missing entities are just reported per mode.

In [0]:
# Matches the entity registry bronze_batch_ingestion.py already ingests.
KNOWN_GTFS_FILES = {
    "agency.txt": "agency",
    "routes.txt": "routes",
    "trips.txt": "trips",
    "stops.txt": "stops",
    "stop_times.txt": "stop_times",
    "calendar.txt": "calendar",
    "calendar_dates.txt": "calendar_dates",
    "shapes.txt": "shapes",
}
 
CRITICAL_ENTITIES = {"trips", "stop_times", "calendar", "routes"}

# Per-mode overrides on top of the base set: buses relies on calendar_dates
# for service exceptions, sydneytrains publishes calendar.txt with no
# exceptions and doesn't need it.
MODE_CRITICAL_OVERRIDES = {
    "buses": CRITICAL_ENTITIES | {"calendar_dates"},
    "sydneytrains": CRITICAL_ENTITIES,
}

def critical_entities_for(mode: str) -> set:
    return MODE_CRITICAL_OVERRIDES.get(mode, CRITICAL_ENTITIES)
 
mode_results = {}  # mode -> {"landed": [...], "skipped": [...], "missing": [...]} or None on hard failure
 
for mode in GTFS_MODES:
    feed_url = GTFS_STATIC_URL_TEMPLATE.format(mode=mode)
    mode_landing_dir = f"{BASE_LANDING}/{mode}"
 
    print(f"--- Mode: {mode} ({feed_url}) ---")
 
    try:
        zip_bytes = download_gtfs_zip(feed_url, api_key)
        print(f"Downloaded {mode} GTFS static bundle: {len(zip_bytes):,} bytes")
 
        landed_entities, skipped_existing, missing_from_feed = [], [], []
 
        with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
            names_in_zip = set(zf.namelist())
 
            for source_name, entity in KNOWN_GTFS_FILES.items():
                if source_name not in names_in_zip:
                    missing_from_feed.append(entity)
                    continue
 
                dest_path = f"{mode_landing_dir}/{entity}_{today_stamp}.csv"
 
                # Don't re-land or overwrite today's file -- Bronze is append-only,
                # so a duplicate landed file becomes duplicate rows.
                try:
                    dbutils.fs.ls(dest_path)
                    skipped_existing.append(entity)
                    continue
                except Exception:
                    pass  # doesn't exist yet -- proceed to write it
 
                with zf.open(source_name) as member:
                    contents = member.read()
 
                dbutils.fs.put(dest_path, contents.decode("utf-8"), overwrite=False)
                landed_entities.append(entity)
 
        mode_results[mode] = {
            "landed": landed_entities,
            "skipped": skipped_existing,
            "missing": missing_from_feed,
        }
        print(f"[{mode}] landed: {sorted(landed_entities)}")
        print(f"[{mode}] already present (skipped): {sorted(skipped_existing)}")
        print(f"[{mode}] not in this feed: {sorted(missing_from_feed)}")
 
    except Exception as e:
        print(f"[{mode}] download/unpack failed entirely: {e}")
        mode_results[mode] = None  # hard failure -- every entity is effectively missing

### 4: Fail loudly if a mode's critical entities didn't land
##### Mirrors Bronze's critical-entity gate, checked per mode -- a buses problem shouldn't be reported as if trains was affected too.

In [0]:
failing_modes = {}
for mode, result in mode_results.items():
    mode_critical = critical_entities_for(mode)
    missing_critical = (
        mode_critical if result is None
        else set(result["missing"]) & mode_critical
    )
    if missing_critical:
        failing_modes[mode] = sorted(missing_critical)
 
if failing_modes:
    raise RuntimeError(
        f"Critical GTFS files missing this run: {failing_modes}. "
        f"Check GTFS_STATIC_URL_TEMPLATE / feed structure before Bronze runs."
    )
 
print("All modes landed their critical entities. Safe to run Bronze.")